In [2]:
import configparser

config = configparser.ConfigParser(interpolation=None)
config.read("config.ini")

import requests
import pandas as pd
from tqdm import tqdm

In [3]:
# Read URL template
url_template = config["API"]["url_path"]

# Read category dictionary
category_codes = {
    "Men's Casual Shoes": config["CATEGORY_CODES"]["mens_casual_shoes"],
    "Women's Casual Footwear": config["CATEGORY_CODES"]["womens_casual_footwear"],
    "Women's Ethnic Footwear": config["CATEGORY_CODES"]["womens_ethnic_footwear"],
    "Women's TShirts": config["CATEGORY_CODES"]["womens_tshirts"],
    "Men's TShirts": config["CATEGORY_CODES"]["mens_tshirts"]
}

destination_path = config["intermediate_path"]["product_table"]

In [4]:
def fetch_and_parse_products(max_products=100, cat_code=None, category=None):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json"
    }

    records = []

    for page in tqdm(range(0, 80)):
        url = url_template.format(cat_code=cat_code, page=page)

        r = requests.get(url, headers=headers)
        data = r.json()

        products = data.get('searchresult', [])

        for p in products:
            try:
                image_url = 'https:' + p["imageURL"]
                image_id = p['productId']
                product_name = p['productname']
                l1_category = p['productCategoryType']
                price = p['price']['sellingPrice']['doubleValue']

                records.append({
                    "image_id": image_id,
                    "image_url": image_url,
                    "l1_category": l1_category,
                    "product_name": product_name,
                    "price": price,
                    "category": category
                })
            except Exception as e:
                continue

            if len(records) >= max_products:
                break

        if len(records) >= max_products:
            break

    return pd.DataFrame(records[:max_products])

In [5]:
dict_ = {"Men's Casual Shoes":'MSH1310113',
"Women's Casual Footwear":'MSH1311113',
"Women's Ethnic Footwear":'MSH1310125',
"Women's TShirts": 'MSH1018100',
"Men's TShirts": 'MSH1116100'}

all_products = pd.DataFrame()
for category, cat_code in dict_.items():
    print('Scraping - ', category)
    df = fetch_and_parse_products(2000, cat_code, category)
    all_products = pd.concat([all_products, df], ignore_index=True)

all_products.head()

Scraping -  Men's Casual Shoes


  0%|          | 0/80 [00:00<?, ?it/s]

 61%|██████▏   | 49/80 [00:15<00:10,  3.10it/s]


Scraping -  Women's Casual Footwear


 62%|██████▎   | 50/80 [00:10<00:06,  4.73it/s]


...

Scraping -  Women's TShirts


 61%|██████▏   | 49/80 [00:10<00:06,  4.72it/s]


Scraping -  Men's TShirts


 61%|██████▏   | 49/80 [00:19<00:12,  2.55it/s]


,image_id,image_url,l1_category,product_name,price,category
0,MP000000029740902,https://img.tatacliq.com/images/i29/437Wx649H/...,Footwear,LUNA BLU by Westside Gold Multi-Strap Slides,999.0,Men's Casual Shoes
1,MP000000022615605,https://img.tatacliq.com/images/i18//437Wx649H...,Footwear,Saint G Women's Matilde Gold T-Strap Sandals,8900.0,Men's Casual Shoes
2,MP000000018260200,https://img.tatacliq.com/images/i12/437Wx649H/...,Footwear,Hush Puppies by Bata Women's Brown Thong Wedges,2999.0,Men's Casual Shoes
3,MP000000028195470,https://img.tatacliq.com/images/i26//437Wx649H...,Footwear,Skechers Women's Summits Mauve Slip-On Shoes,3839.0,Men's Casual Shoes
4,MP000000024268627,https://img.tatacliq.com/images/i20//437Wx649H...,Footwear,Metro Women's Brown Wedge Pumps,1748.5,Men's Casual Shoes


In [6]:
all_products.category.value_counts()


category
Men's Casual Shoes         2000
Women's Casual Footwear    2000
Women's Ethnic Footwear    2000
Women's TShirts            2000
Men's TShirts              2000
Name: count, dtype: int64

In [7]:
all_products.to_csv(destination_path, index=False)